# Chapter 3 – Content-Based Embeddings with Sentence Transformers

This notebook implements Sections 3.4.1 through 3.4.3:
- Creating content embeddings with Sentence Transformers
- Finding similar movies by embedding similarity
- Handling multiple seed items (average vs max aggregation)

In [ ]:
%pip install sentence-transformers
%pip install python-dotenv

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from recsys.data.loaders import (
    load_movielens,
    load_movielens_links,
    load_tmdb_movie_descriptions,
)
from recsys.utils.colab import get_data_path

## 1. Load data and movie descriptions

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
tmdb_api_key = os.getenv("TMDB_API_KEY")

DATA_PATH = get_data_path()
ratings, movies = load_movielens("ml-25m", data_dir=DATA_PATH)
links = load_movielens_links("ml-25m", data_dir=DATA_PATH)

In [ ]:
# Fetch descriptions from TMDB (cached after first run)
descriptions = load_tmdb_movie_descriptions(
    links=links,
    api_key=tmdb_api_key,
    data_dir=DATA_PATH,
    cache_filename="movielens_descriptions.csv",
    force_refresh=False,
)

# Merge descriptions into movies DataFrame
desc_df = pd.DataFrame.from_dict(descriptions, orient='index')
desc_df.reset_index(names='movieId', inplace=True)
desc_df['movieId'] = desc_df['movieId'].astype(str)

movies = movies.merge(desc_df[['movieId', 'overview']], on='movieId', how='left')
movies['overview'] = movies['overview'].fillna('')

## 2. Prepare text and generate embeddings (Listings 3.11–3.12)

In [ ]:
# Listing 3.11: Preparing the text
movies['content'] = (
    movies['title'] + ' ' +
    movies['genres'] + ' ' +
    movies['overview'].fillna('')
)  #A

#A Combine title, genres, and description into a single text

In [ ]:
# Listing 3.12: Create embeddings for the movies
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  #A

embeddings = model.encode(
    movies['content'].tolist(),
    show_progress_bar=True,
    batch_size=32
)  #B

#A Load the model
#B Generate embeddings in batches for efficiency

print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dtype: {embeddings.dtype}")

In [ ]:
# Listing 3.13: Looking at embeddings
toy_story_idx = 0
print(f"Toy Story embedding (first 10 dimensions):")
print(embeddings[toy_story_idx][:10])
print(f"\nMagnitude: {np.linalg.norm(embeddings[toy_story_idx]):.3f}")

## 3. Finding similar items (Listing 3.14)

In [ ]:
# Listing 3.14: Finding similar movies by embedding similarity
def find_similar_movies(movie_idx, embeddings, k=10):
    query_emb = embeddings[movie_idx:movie_idx+1]  #A
    similarities = cosine_similarity(query_emb, embeddings)[0]  #B
    similarities[movie_idx] = -1  #C
    top_indices = np.argsort(similarities)[-(k):][::-1]  #D
    return top_indices, similarities[top_indices]

#A Keep as 2D array for sklearn
#B Compute similarity to all movies at once
#C Exclude the query movie itself
#D Get indices of top k most similar

In [ ]:
def print_similar(indices, scores, movies_df):
    """Display similar movies with scores and genres."""
    for i, (idx, score) in enumerate(zip(indices, scores), 1):
        movie = movies_df.iloc[idx]
        print(f"{i}. {movie['title']} sim: {score:.3f}, Genres: {movie['genres']}")


indices, scores = find_similar_movies(0, embeddings, k=5)  # Toy Story
print("Movies similar to Toy Story (1995):\n")
print_similar(indices, scores, movies)

## 4. Multiple seed items (Listing 3.15)

The same aggregation trade-offs from Section 3.3.6 apply here.
We show both averaging (centroid) and max aggregation.

In [ ]:
# Listing 3.15: Finding items similar to multiple seeds (average)
def find_similar_to_multiple(seed_indices, embeddings, k=10):
    seed_embeddings = embeddings[seed_indices]  #A
    avg_embedding = np.mean(seed_embeddings, axis=0, keepdims=True)  #B
    similarities = cosine_similarity(avg_embedding, embeddings)[0]  #C
    for idx in seed_indices:
        similarities[idx] = -1  #D
    top_indices = np.argsort(similarities)[-(k):][::-1]
    return top_indices, similarities[top_indices]

#A Get embeddings for all seed items
#B Average them into a single query embedding
#C Find items similar to the average
#D Exclude seed items from results

In [ ]:
def find_similar_max_aggregation(seed_indices, embeddings, k=10):
    """Alternative: per-seed retrieval with max merge (Section 3.3.6)."""
    all_similarities = {}
    for seed_idx in seed_indices:
        query_emb = embeddings[seed_idx:seed_idx+1]
        similarities = cosine_similarity(query_emb, embeddings)[0]
        for idx, score in enumerate(similarities):
            if idx in seed_indices:
                continue
            if idx not in all_similarities or score > all_similarities[idx]:
                all_similarities[idx] = score
    sorted_items = sorted(all_similarities.items(), key=lambda x: x[1], reverse=True)
    top = sorted_items[:k]
    return np.array([x[0] for x in top]), np.array([x[1] for x in top])

In [ ]:
# Seeds: Toy Story (0), Jumanji (1), Grumpier Old Men (2)
seed_indices = [0, 1, 2]
seed_titles = [movies.iloc[i]['title'] for i in seed_indices]
print(f"Seeds: {seed_titles}\n")

print("--- Approach 1: Average embeddings ---")
avg_idx, avg_scores = find_similar_to_multiple(seed_indices, embeddings, k=5)
print_similar(avg_idx, avg_scores, movies)

print("\n--- Approach 2: Max aggregation ---")
max_idx, max_scores = find_similar_max_aggregation(seed_indices, embeddings, k=5)
print_similar(max_idx, max_scores, movies)

## 5. Cache embeddings for reuse

Embeddings are expensive to compute but cheap to store.

In [ ]:
from pathlib import Path

cache_path = Path(DATA_PATH) / "movie_embeddings_minilm.npz"

# Save
np.savez_compressed(
    cache_path,
    embeddings=embeddings,
    movie_ids=movies['movieId'].values
)
print(f"Saved embeddings to {cache_path}")
print(f"File size: {cache_path.stat().st_size / (1024**2):.1f} MB")

# Load (for subsequent runs)
# cache = np.load(cache_path, allow_pickle=True)
# embeddings = cache['embeddings']
# movie_ids = cache['movie_ids']